# Physical Concepts Are Linearly Represented

We use linear probes to test whether meaningful jet properties like jet mass, particle multiplicity, jet charge, and track displacement are linearly decodable from the model's internal representations.

For each jet characteristic and transformer block, we train a separate ridge-regression model to predict the concept from the CLS representation at the output of the corresponding block. All probes use the same 8,000 QCD jets for training and 2,000 QCD jets for held-out evaluation, sampled from the JetClass validation set.

In [1]:
import csv
import json
import warnings

from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import joblib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
from IPython.display import Markdown, display

from jetdirections.config import (
    CHECKPOINT_DIR,
    DROPOUT,
    MODEL_DIM,
    NUM_CLASSES,
    NUM_HEADS,
    NUM_LAYERS,
    PROJ_ROOT,
    VAL_DATA_DIR,
)
from jetdirections.data.dataset import JetClassDataset
from jetdirections.data.sampling import load_class_batch
from jetdirections.data.vectorize import FEATURE_NAMES
from jetdirections.interp.activations import extract_cls_activations
from jetdirections.interp.probes import DEFAULT_RIDGE_ALPHAS, fit_linear_probe
from jetdirections.model.transformer import JetTagTransformer

N_JETS = 10_000
DATA_SEED = 1
SPLIT_SEED = 0
LAYERS = list(range(NUM_LAYERS))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = PROJ_ROOT / "experiments" / "results" / "physical_concept_probes"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device: {DEVICE}")
print(f"torch: {torch.__version__}; sklearn: {sklearn.__version__}")
print(f"ridge penalties: {DEFAULT_RIDGE_ALPHAS}")

device: cuda
torch: 2.12.0+cu130; sklearn: 1.9.0
ridge penalties: [1.00000000e-02 3.16227766e-02 1.00000000e-01 3.16227766e-01
 1.00000000e+00 3.16227766e+00 1.00000000e+01 3.16227766e+01
 1.00000000e+02 3.16227766e+02 1.00000000e+03 3.16227766e+03
 1.00000000e+04]


# Setup

### Load Model

In [2]:
checkpoint_path = CHECKPOINT_DIR / "jetclass_transformer.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

model = JetTagTransformer(
    input_dim=17,
    embed_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
).to(DEVICE)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state)
model.eval();

### Load Dataset

In [3]:
dataset = JetClassDataset(VAL_DATA_DIR, shuffle_files=False)
qcd = load_class_batch(dataset, "QCD", batch_size=N_JETS, seed=DATA_SEED)
x, mask, labels = qcd

assert len(x) == N_JETS
assert labels.eq(0).all(), "The probe sample must contain only QCD jets."
print(f"x: {tuple(x.shape)}, mask: {tuple(mask.shape)}, labels: {tuple(labels.shape)}")

x: (10000, 17, 128), mask: (10000, 128), labels: (10000,)


### Load CLS Activations to Train/Evaluate Linear Probe On

In [4]:
# Load the CLS representation of each QCD jet at the output of every block.
print(
    f"Expected: {N_JETS:,} activations per block, one for each QCD jet; "
    f"each activation has dimension {MODEL_DIM}."
)
activations = extract_cls_activations(model, qcd, LAYERS, chunk_size=512)
for layer in LAYERS:
    assert activations[layer].shape == (N_JETS, MODEL_DIM)
    print(f"Block {layer} activations shape: {tuple(activations[layer].shape)}")

permutation = torch.randperm(
    N_JETS, generator=torch.Generator().manual_seed(SPLIT_SEED)
).numpy()
n_test = N_JETS // 5
test_indices = permutation[:n_test]
train_indices = permutation[n_test:]
assert not np.intersect1d(train_indices, test_indices).size
print(f"Training examples: {len(train_indices):,}")
print(f"Held-out examples: {len(test_indices):,}")

Expected: 10,000 activations per block, one for each QCD jet; each activation has dimension 512.
Block 0 activations shape: (10000, 512)
Block 1 activations shape: (10000, 512)
Block 2 activations shape: (10000, 512)
Block 3 activations shape: (10000, 512)
Block 4 activations shape: (10000, 512)
Block 5 activations shape: (10000, 512)
Training examples: 8,000
Held-out examples: 2,000


### Notebook Helper Functions

In [5]:
def fit_at_all_blocks(target):
    """Fit one probe per block for a target with one value per jet.

    Returns a dictionary mapping each block index to a LinearProbeResult.
    """
    target = np.asarray(target, dtype=np.float64)
    return {
        layer: fit_linear_probe(
            activations[layer].numpy(),
            target,
            train_indices,
            test_indices,
        )
        for layer in LAYERS
    }


def display_block_results(results):
    lines = ["| Block | Ridge alpha | Train $R^2$ | Held-out $R^2$ |",
             "|---:|---:|---:|---:|"]
    for layer, result in results.items():
        lines.append(
            f"| {layer} | {result.alpha:g} | {result.train_r2:.4f} | {result.test_r2:.4f} |"
        )
    display(Markdown("\n".join(lines)))

# Experiments

## Jet Mass

In [6]:
PT_LOG = FEATURE_NAMES.index("part_pt_log")
DETA = FEATURE_NAMES.index("part_deta")
DPHI = FEATURE_NAMES.index("part_dphi")


def jet_mass(x, mask):
    """Invariant mass in GeV from massless constituents; mask=True means padding."""
    pt = torch.exp(x[:, PT_LOG] / 0.7 + 1.7) * (~mask)
    deta, dphi = x[:, DETA], x[:, DPHI]
    px = (pt * torch.cos(dphi)).sum(dim=1)
    py = (pt * torch.sin(dphi)).sum(dim=1)
    pz = (pt * torch.sinh(deta)).sum(dim=1)
    energy = (pt * torch.cosh(deta)).sum(dim=1)
    return torch.sqrt(torch.clamp(energy**2 - px**2 - py**2 - pz**2, min=0.0))


mass = jet_mass(x, mask)
log_mass = torch.log(mass.clamp(min=1.0)).numpy()
print(f"mass: {mass.mean():.1f} +/- {mass.std():.1f} GeV "
      f"(min {mass.min():.1f}, max {mass.max():.1f})")
mass_results = fit_at_all_blocks(log_mass)
display_block_results(mass_results)

mass: 84.8 +/- 55.0 GeV (min 0.0, max 414.0)


| Block | Ridge alpha | Train $R^2$ | Held-out $R^2$ |
|---:|---:|---:|---:|
| 0 | 0.316228 | 0.9932 | 0.9916 |
| 1 | 3.16228 | 0.9925 | 0.9916 |
| 2 | 3.16228 | 0.9903 | 0.9887 |
| 3 | 10 | 0.9882 | 0.9857 |
| 4 | 31.6228 | 0.9870 | 0.9839 |
| 5 | 31.6228 | 0.9864 | 0.9833 |

## Particle Multiplicity

In [7]:
def particle_multiplicity(mask):
    """Number of real particle tokens in each jet."""
    return (~mask).sum(dim=1)


multiplicity = particle_multiplicity(mask).float()
print(f"multiplicity: {multiplicity.mean():.1f} +/- {multiplicity.std():.1f} "
      f"(min {multiplicity.min():.0f}, max {multiplicity.max():.0f})")
multiplicity_results = fit_at_all_blocks(multiplicity.numpy())
display_block_results(multiplicity_results)

multiplicity: 35.3 +/- 17.1 (min 1, max 127)


| Block | Ridge alpha | Train $R^2$ | Held-out $R^2$ |
|---:|---:|---:|---:|
| 0 | 0.0316228 | 0.9980 | 0.9978 |
| 1 | 1 | 0.9967 | 0.9964 |
| 2 | 3.16228 | 0.9948 | 0.9945 |
| 3 | 10 | 0.9928 | 0.9921 |
| 4 | 31.6228 | 0.9909 | 0.9902 |
| 5 | 31.6228 | 0.9898 | 0.9892 |

## Jet Charge

In [8]:
CHARGE = FEATURE_NAMES.index("part_charge")


def jet_charge(x, mask):
    """pT-weighted constituent charge with kappa=1."""
    pt = torch.exp(x[:, PT_LOG] / 0.7 + 1.7) * (~mask)
    charge = x[:, CHARGE] * (~mask)
    return (charge * pt).sum(dim=1) / pt.sum(dim=1).clamp(min=1e-8)


charge = jet_charge(x, mask)
print(f"jet charge: {charge.mean():+.3f} +/- {charge.std():.3f} "
      f"(min {charge.min():+.3f}, max {charge.max():+.3f})")
charge_results = fit_at_all_blocks(charge.numpy())
display_block_results(charge_results)

jet charge: +0.024 +/- 0.207 (min -0.926, max +0.900)


| Block | Ridge alpha | Train $R^2$ | Held-out $R^2$ |
|---:|---:|---:|---:|
| 0 | 0.1 | 0.9832 | 0.9812 |
| 1 | 1 | 0.9793 | 0.9752 |
| 2 | 10 | 0.9702 | 0.9668 |
| 3 | 31.6228 | 0.9590 | 0.9557 |
| 4 | 31.6228 | 0.9518 | 0.9458 |
| 5 | 100 | 0.9464 | 0.9401 |

## Track Displacement

In [9]:
D0 = FEATURE_NAMES.index("part_d0")
D0ERR = FEATURE_NAMES.index("part_d0err")


def track_displacement(x, mask):
    """log(1 + summed |d0|/sigma(d0)) over charged real particles."""
    charged = (x[:, CHARGE] != 0) & (~mask)
    d0 = torch.atanh(x[:, D0].clamp(-0.999999, 0.999999))
    significance = d0.abs() / x[:, D0ERR].clamp(min=1e-6)
    return torch.log1p((significance * charged).sum(dim=1))


displacement = track_displacement(x, mask)
print(f"track displacement: {displacement.mean():.2f} +/- {displacement.std():.2f} "
      f"(min {displacement.min():.2f}, max {displacement.max():.2f})")
displacement_results = fit_at_all_blocks(displacement.numpy())
display_block_results(displacement_results)

track displacement: 4.06 +/- 1.80 (min 0.00, max 8.05)


| Block | Ridge alpha | Train $R^2$ | Held-out $R^2$ |
|---:|---:|---:|---:|
| 0 | 0.316228 | 0.8760 | 0.8754 |
| 1 | 3.16228 | 0.8766 | 0.8750 |
| 2 | 10 | 0.8561 | 0.8579 |
| 3 | 31.6228 | 0.8380 | 0.8388 |
| 4 | 100 | 0.8306 | 0.8300 |
| 5 | 31.6228 | 0.8270 | 0.8256 |

## Combined Results

In [10]:
all_results = {
    "Jet Mass": mass_results,
    "Particle Multiplicity": multiplicity_results,
    "Jet Charge": charge_results,
    "Track Displacement": displacement_results,
}
concept_definitions = {
    "Jet Mass": r"$\log(\max(\sqrt{E^2-p_x^2-p_y^2-p_z^2},1\,\mathrm{GeV}))$",
    "Particle Multiplicity": r"$N=\sum_i \mathbf{1}_i$",
    "Jet Charge": r"$Q=\sum_i q_i p_{T,i}/\sum_i p_{T,i}$",
    "Track Displacement": r"$D=\log(1+\sum_{i\in\mathrm{ch}}|d_{0,i}|/\sigma(d_{0,i}))$",
}

summary_lines = [
    "| Concept | Definition / probe target | Held-out probe $R^2$ (blocks 0–5) |",
    "|:--|:--|:--|",
]
for concept, results in all_results.items():
    scores = [results[layer].test_r2 for layer in LAYERS]
    summary_lines.append(
        f"| {concept} | {concept_definitions[concept]} | {min(scores):.3f}–{max(scores):.3f} |"
    )
summary_table = "\n".join(summary_lines)
display(Markdown(summary_table))

| Concept | Definition / probe target | Held-out probe $R^2$ (blocks 0–5) |
|:--|:--|:--|
| Jet Mass | $\log(\max(\sqrt{E^2-p_x^2-p_y^2-p_z^2},1\,\mathrm{GeV}))$ | 0.983–0.992 |
| Particle Multiplicity | $N=\sum_i \mathbf{1}_i$ | 0.989–0.998 |
| Jet Charge | $Q=\sum_i q_i p_{T,i}/\sum_i p_{T,i}$ | 0.940–0.981 |
| Track Displacement | $D=\log(1+\sum_{i\in\mathrm{ch}}|d_{0,i}|/\sigma(d_{0,i}))$ | 0.826–0.875 |

## Save Fitted Probes

Persist the fitted `RidgeCV` probes themselves (not just their scores) so downstream notebooks — e.g. the steering-persistence experiment in `04_physical_concept_downstream_probes.ipynb` — read out the exact same probes reported here, rather than refitting a nominally-identical but distinct copy.

In [11]:
probes_path = RESULTS_DIR / "fitted_probes.joblib"
joblib.dump(all_results, probes_path)
print(f"Saved fitted probes for {list(all_results)} (blocks {LAYERS}) to {probes_path}")

Saved fitted probes for ['Jet Mass', 'Particle Multiplicity', 'Jet Charge', 'Track Displacement'] (blocks [0, 1, 2, 3, 4, 5]) to <project-root>/experiments/results/physical_concept_probes/fitted_probes.joblib
